# Michigan Traders: Module 8
# Machine Learning for Alpha  ·  *interactive workbook*

**Series:** MAT Education · ML & Capstone
**Level:** Intermediate (builds on Modules 1–7)
**Format:** **Guided + Your-Turn**, with self-checking exercises you can run on your own laptop.

---

Every module so far has ended with the same warning in a different costume. Module 6: a strategy found by searching 200 candidates is not a strategy. Module 7: 1,770 pair tests produce dozens of discoveries in data containing none.

Machine learning is that problem with the safety off.

A random forest with default settings will fit **any** target you hand it, perfectly, including pure noise. It will then report an accuracy that depends entirely on how you split your data, and the split that everyone reaches for first, the one in every tutorial, is *wrong for time series* in a way that silently inflates the result.

You are about to see it happen. The same model on the same data will score **72.0%** under the standard split and **50.5%** under the correct one. That is the whole module in two numbers: one of them is a discovery and the other is a coin flip, and nothing about the code tells you which is which.

| | |
|---|---|
| **What ML is good at here** | combining many weak signals; finding interactions you would not hand-code |
| **What it is bad at** | low signal-to-noise data, non-stationary relationships, and telling you when it has learned nothing |
| **What this module teaches** | how to get a number you are allowed to believe |

## How to use this notebook

| Cell type | Where it runs | What to do |
|---|---|---|
| 🟢 **Local cell** | your laptop's Jupyter | Run it. Output is baked in so you can read along. |
| 🔵 **QC cell** | QuantConnect (LEAN) | Copy into a research notebook or algorithm. It will *not* run locally. |

**New dependency.** This module uses `scikit-learn`, pre-installed on QuantConnect. Locally:

```
pip install scikit-learn
```

Answers are in `08_Machine_Learning_for_Alpha_SOLUTIONS.ipynb`.

> ⚠️ **A word on the data.** As in Module 7, we simulate a market so that we know the truth. This one has a *genuine* predictable component built into it, a hidden regime that persists, which is what lets us show the difference between a model that has found something and a model that only appears to have. Real markets are far less generous than this. Treat every accuracy figure below as an upper bound on what the same code would do on real data.

## 1. What we are actually predicting

Before any code, a framing decision that beginners usually get wrong.

**Do not predict the price.** "Tomorrow's price will be 102.37" is a regression problem with a trivial, useless solution: today's price. A model that predicts tomorrow's price from today's will score a magnificent R² of 0.99 and contain exactly zero information, because it has learned "prices barely change day to day". You cannot trade it.

**Predict something you can act on.** The two usual choices:

| Target | Type | Trade |
|---|---|---|
| *Will the return over the next N days be positive?* | **classification** | long if yes, short/flat if no |
| *How large will the next N days' return be?* | **regression** | size the position by the prediction |

We will use the first: a **binary classification** of direction over the next 5 days. It maps directly onto a position, which is what Module 6 needs to evaluate anything.

### The vocabulary

If you have not done supervised learning before, this is the entire conceptual load:

| Term | Meaning | Here |
|---|---|---|
| **features** (`X`) | the inputs; what the model gets to look at | momentum, volatility, moving-average ratio… |
| **target** (`y`) | the output; what it is asked to predict | 1 if the next 5 days are up, else 0 |
| **training set** | rows the model learns from | the earlier part of history |
| **test set** | rows it has never seen, used to score it | the later part |
| **fit** | the learning step | `model.fit(X_train, y_train)` |
| **predict** | apply the learned rule | `model.predict(X_test)` |
| **accuracy** | share of test rows predicted correctly | 0.5 is a coin flip |

The single idea holding it together: **a model is only worth what it scores on data it has never seen.** Everything that goes wrong in this module is a violation of that sentence, usually by accident.

### Setup: a market with something to find

The hidden `state` flips between +1 and −1 at random (a 1.2% chance on any given day, so a regime runs about 60 business days), and adds a small drift in whichever direction it currently points. That drift is tiny next to daily noise, 0.16% against 1.1%, but it *persists*, which is what makes it learnable in principle.

A student is not told the state exists. Neither is the model. Both see only prices.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 4)

rng = np.random.default_rng(5)
n = 1512                                     # ~6 trading years
dates = pd.bdate_range("2018-01-02", periods=n)

DRIFT, SWITCH, VOL = 0.0016, 0.012, 0.011

state = np.zeros(n)
s = 1
for t in range(n):
    if rng.random() < SWITCH:                # ~1.2% chance of flipping each day
        s = -s
    state[t] = s

returns = DRIFT * state + rng.normal(0, VOL, n)
price = pd.Series(100 * np.exp(np.cumsum(returns)), index=dates, name="close")
regime = pd.Series(state, index=dates, name="regime")

print(f"price {price.iloc[0]:.1f} -> {price.iloc[-1]:.1f} over {n} days")
print(f"regime switches      : {int((regime.diff() != 0).sum() - 1)}")
print(f"mean days per regime : {n / (regime.diff() != 0).sum():.0f}")
print(f"daily drift {DRIFT:.4f} vs daily noise {VOL:.4f} "
      f"-> signal is {DRIFT / VOL:.2f}x the noise")

price 98.0 -> 125.0 over 1512 days
regime switches      : 24
mean days per regime : 60
daily drift 0.0016 vs daily noise 0.0110 -> signal is 0.15x the noise


A signal 0.15× the size of the noise. That ratio is the reason financial ML is hard, and it is *generous* compared to reality. Over one day the drift is invisible. Over the 80-day life of a regime it accumulates to something a model can, in principle, detect.

The whole game is detecting it without also "detecting" a hundred things that are not there.

## 2. Building the feature matrix

Features are just columns, and every one of them is something you already built in Module 5. What changes is that the model, not you, decides how to combine them.

Six features, each a different view of the same price series:

| Feature | What it measures |
|---|---|
| `mom_5`, `mom_20`, `mom_60` | return over the last 5 / 20 / 60 days |
| `vol_20` | 20-day realised volatility |
| `range_pos` | where price sits in its 14-day range, 0 = low, 1 = high |
| `ma_ratio` | 10-day mean ÷ 50-day mean; above 1 means the fast average leads |

Every one is computed from data available **at that moment**. That property is not automatic and it is the thing to check first in anyone's feature code, including your own.

In [2]:
daily = price.pct_change()

features = pd.DataFrame({
    "mom_5": price.pct_change(5),
    "mom_20": price.pct_change(20),
    "mom_60": price.pct_change(60),
    "vol_20": daily.rolling(20).std(),
    "range_pos": ((price - price.rolling(14).min())
                  / (price.rolling(14).max() - price.rolling(14).min())),
    "ma_ratio": price.rolling(10).mean() / price.rolling(50).mean(),
})

print(features.shape)
features.iloc[[60, 500, 1000]].round(4)

(1512, 6)


,mom_5,mom_20,mom_60,vol_20,range_pos,ma_ratio
2018-03-27,-0.0277,-0.0111,0.0287,0.0164,0.1040,0.9918
2019-12-03,-0.0165,0.0008,-0.0132,0.0144,0.3316,0.9885
2021-11-02,-0.0135,-0.0806,-0.0017,0.0130,0.3496,0.9305


### The target, and the one line that decides whether any of this is real

The target is the direction of the **next** 5 days:

```python
forward = price.shift(-5) / price - 1
target = (forward > 0).astype(int)
```

`shift(-5)` pulls future prices backwards to sit beside today's features. That is correct here (we *want* to line up "what happened next" with "what we knew then"), and it is also the most dangerous line in the notebook, because the same mechanic applied to a feature is look-ahead bias.

> 🧠 **The rule: `shift(-k)` is allowed in the target and forbidden in the features.** If a negative shift ever appears in a feature column, that feature can see the future and the entire result is void.

Note also that the last 5 rows have no future to look at, so the target is `NaN` there and the rows are dropped.

In [3]:
HORIZON = 5

forward = price.shift(-HORIZON) / price - 1
target = (forward > 0).astype(int).rename("target")

data = features.join(target).dropna()
X = data.drop(columns="target")
y = data["target"]

print(f"rows after dropping NaN : {len(data)}  (from {len(features)})")
print(f"share of 'up' periods   : {y.mean():.3f}")
print(f"date range              : {data.index[0].date()} to {data.index[-1].date()}")

rows after dropping NaN : 1452  (from 1512)
share of 'up' periods   : 0.503
date range              : 2018-03-27 to 2023-10-18


A 50.3% up-rate is the ideal starting point for teaching: the classes are balanced, so accuracy is easy to read and a naive "always predict up" model would score about 50%. Real data is rarely this tidy, which is why section 8 measures the baseline rather than assuming it.

### ✏️ Your turn: build features and a target

Write two functions.

`make_features(px)` returns a DataFrame with exactly these four columns, in this order:

| column | definition |
|---|---|
| `mom_10` | 10-day percentage change of `px` |
| `vol_10` | 10-day rolling std of daily percentage change |
| `ma_ratio_5_20` | 5-day mean ÷ 20-day mean |
| `range_pos_10` | `(px − 10-day min) / (10-day max − 10-day min)` |

`make_target(px, horizon)` returns an int Series: `1` where the return over the next `horizon` days is positive, else `0`, with the trailing `NaN` rows **dropped**.

Then build `X2` and `y2` by joining them with `horizon=3` and dropping any row with a `NaN`, exactly as above. `X2` must contain only the four feature columns.

In [ ]:
def make_features(px):
    d = px.pct_change()
    return pd.DataFrame({
        "mom_10": ...,           # TODO
        "vol_10": ...,           # TODO
        "ma_ratio_5_20": ...,    # TODO
        "range_pos_10": ...,     # TODO
    })


def make_target(px, horizon):
    # TODO: 1 if the forward return is positive, else 0; drop the trailing NaNs
    ...


f2 = make_features(price)
t2 = make_target(price, 3).rename("target")

d2 = ...
X2 = ...
y2 = ...

print(X2.shape, f"up-rate {y2.mean():.3f}")

In [ ]:
COLS = ["mom_10", "vol_10", "ma_ratio_5_20", "range_pos_10"]
assert list(X2.columns) == COLS, f"columns must be exactly {COLS} in order, got {list(X2.columns)}"
assert len(X2) == len(y2), "X2 and y2 must have the same number of rows"
assert not X2.isna().any().any(), "X2 still contains NaN"
assert set(y2.unique()) <= {0, 1}, "the target must be 0/1 integers"
# The target must look FORWARD, not backward.
_fwd = price.shift(-3) / price - 1
_expect = (_fwd > 0).astype(int)[_fwd.notna()]
assert y2.equals(_expect.loc[y2.index].astype(y2.dtype)), \
    "y2 is off - the target is the sign of the return over the NEXT 3 days"
# The last 3 dates have no future and must not survive.
assert price.index[-1] not in y2.index, \
    "the final rows have no forward return and must be dropped, not filled"
# Features must be backward-looking only: rebuilding on a truncated series must
# leave the earlier values untouched.
_trunc = make_features(price.iloc[:800])
_full = make_features(price).iloc[:800]
assert np.allclose(_trunc.dropna().values, _full.loc[_trunc.dropna().index].values), \
    ("a feature computed on a truncated series changed - that means a feature is "
     "looking ahead. Check for shift(-k) or centered windows.")
print(f"✅ Correct!  {X2.shape[0]} rows x {X2.shape[1]} features, up-rate {y2.mean():.3f},",
      "and every feature is backward-looking")

## Part 1: How the standard approach lies to you

## 3. The tutorial version

Here is what almost every ML tutorial tells you to do, and what you would write if you had not read this module. `train_test_split` holds out 30% of the rows at random to test on.

We will use a **random forest**: a collection of decision trees, each one a flowchart of "if `mom_20` is above this, and `vol_20` is below that, then predict up". Each tree sees a random subset of rows and columns, and the forest averages their votes. It is the standard first choice for tabular data. It handles interactions automatically and needs no feature scaling.

Read the output before reading anything else.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle=True, random_state=0)

model = RandomForestClassifier(n_estimators=200, random_state=0)
model.fit(X_train, y_train)

print(f"training accuracy : {model.score(X_train, y_train):.3f}")
print(f"test accuracy     : {model.score(X_test, y_test):.3f}")

training accuracy : 1.000
test accuracy     : 0.720


**72.0% accuracy at predicting 5-day market direction.**

If that number were real it would be one of the most valuable results in finance. A coin flip is 50%; the entire hedge fund industry runs on edges of a few percent. Twenty-two points of edge would be a fortune.

It is not real. Nothing about the code is wrong in the sense of raising an error, and there is no bug to find. The result is an artefact of `shuffle=True`.

Change that one argument to a time-ordered split (train on the first 70% of history, test on the last 30%, which is the only thing you could actually have done), and watch.

In [7]:
split = int(len(X) * 0.7)
X_tr, X_te = X.iloc[:split], X.iloc[split:]
y_tr, y_te = y.iloc[:split], y.iloc[split:]

model_time = RandomForestClassifier(n_estimators=200, random_state=0)
model_time.fit(X_tr, y_tr)

print(f"training accuracy : {model_time.score(X_tr, y_tr):.3f}")
print(f"test accuracy     : {model_time.score(X_te, y_te):.3f}")
print()
print(f"shuffled split said : {model.score(X_test, y_test):.3f}")
print(f"honest split says   : {model_time.score(X_te, y_te):.3f}")

training accuracy : 1.000
test accuracy     : 0.505

shuffled split said : 0.720
honest split says   : 0.505


**50.5%.** A coin flip.

Same data, same features, same model, same random seed. The only difference is which rows were allowed into the training set, and it moved the headline result by 21.5 percentage points.

This is not a subtle statistical concern to note in a footnote. It is the difference between a strategy and nothing at all, and it is the single most common fatal error in student ML-for-trading projects.

Notice something else in both outputs: **training accuracy is 1.000**. The forest has memorised its training rows perfectly. Hold that thought. It is section 7.

## 4. Why shuffling leaks

Two mechanisms, working together. Both come from the fact that a time series is not a bag of independent rows.

### Mechanism 1: the targets overlap

Our target is the return over the next **5** days. So the target for Monday covers Tuesday through Monday-next; the target for Tuesday covers Wednesday through Tuesday-next. They share four of their five days.

Monday and Tuesday are therefore near-copies of each other. A shuffled split cheerfully puts Monday in training and Tuesday in test, then congratulates the model for predicting Tuesday.

### Mechanism 2: the features barely move

A 60-day momentum changes by roughly 1/60th of its content each day. A 50-day moving average ratio is even slower. Consecutive rows of `X` are nearly identical points.

In [8]:
acf = pd.DataFrame({
    "lag-1 autocorrelation": [X[c].autocorr(1) for c in X.columns]
}, index=X.columns).sort_values("lag-1 autocorrelation", ascending=False)

print(acf.round(4))
print()
print(f"target lag-1 autocorrelation: {y.autocorr(1):.4f}")

           lag-1 autocorrelation
ma_ratio                  0.9978
mom_60                    0.9929
mom_20                    0.9742
vol_20                    0.9445
range_pos                 0.8698
mom_5                     0.8433

target lag-1 autocorrelation: 0.6513


`ma_ratio` is 0.998 correlated with its own value yesterday. `mom_60` is 0.993. Even the fastest feature, `mom_5`, sits at 0.84. These are not 1,452 independent observations; they are a few dozen genuinely distinct market situations, each sampled many times.

So "train on 70% of the rows, test on the other 30%" does not hold out 30% of the *information*. It holds out almost none of it.

### Measuring the leak directly

Here is the cleanest way to see it. For each test row, find the closest training row (literally, the nearest point in feature space), and ask how far away it is, and **how many days apart the two rows are in calendar time**.

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# Put all features on the same scale so distances are comparable.
Z = pd.DataFrame(StandardScaler().fit_transform(X), index=X.index, columns=X.columns)
row_number = {d: i for i, d in enumerate(X.index)}


def leak_report(train_idx, test_idx, label):
    nn = NearestNeighbors(n_neighbors=1).fit(Z.loc[train_idx].values)
    dist, which = nn.kneighbors(Z.loc[test_idx].values)

    # How many days separate each test row from its nearest training row?
    gaps = np.array([abs(row_number[test_idx[i]] - row_number[train_idx[which[i][0]]])
                     for i in range(len(test_idx))])

    print(f"{label}")
    print(f"   median distance to nearest training row : {np.median(dist):.4f}")
    print(f"   nearest training row is within 2 days   : {(gaps <= 2).mean():.1%}")


leak_report(X_train.index, X_test.index, "SHUFFLED split")
print()
leak_report(X_tr.index, X_te.index, "TIME-ORDERED split")

SHUFFLED split
   median distance to nearest training row : 0.4484
   nearest training row is within 2 days   : 50.9%

TIME-ORDERED split
   median distance to nearest training row : 0.6552
   nearest training row is within 2 days   : 0.2%


Under the shuffled split, **half of every test row's nearest neighbour in the training set is a row from within two days of it.** The model is not predicting the future. It is looking up an almost identical row it was trained on and reading off the answer, which is a memory test, not a forecast.

Under the time-ordered split there is a gap between train and test, so nothing is close by, and the accuracy falls to what the model actually knows: nothing much.

> ⚠️ **Never use `shuffle=True` on time-series data.** `train_test_split` defaults to `shuffle=True`. `cross_val_score` and `KFold` shuffle by default too. The library is built for independent rows and will not warn you.

### ✏️ Your turn: quantify the near-duplicate problem

Write `neighbour_gap_share(train_idx, test_idx, within=2)` returning the **fraction** of test rows whose nearest training row (in the scaled feature space `Z`) lies within `within` days of them, as a float.

Reuse the `Z` frame and the `row_number` lookup from the cell above; the row gap is the absolute difference in row numbers.

Then compute:

| variable | value |
|---|---|
| `gap_shuffled` | the share for the shuffled split (`X_train.index`, `X_test.index`) |
| `gap_time` | the share for the time-ordered split (`X_tr.index`, `X_te.index`) |

A leaky split is one where this number is large.

In [ ]:
def neighbour_gap_share(train_idx, test_idx, within=2):
    nn = NearestNeighbors(n_neighbors=1).fit(Z.loc[train_idx].values)
    dist, which = nn.kneighbors(Z.loc[test_idx].values)

    # TODO: row-number gap between each test row and its nearest training row
    ...


gap_shuffled = neighbour_gap_share(X_train.index, X_test.index)
gap_time = neighbour_gap_share(X_tr.index, X_te.index)

print(f"shuffled {gap_shuffled:.1%} | time-ordered {gap_time:.1%}")

In [ ]:
assert 0.0 <= gap_shuffled <= 1.0 and 0.0 <= gap_time <= 1.0, \
    "both results should be fractions between 0 and 1"
assert gap_shuffled > 0.35, \
    f"the shuffled split should leave most test rows with a very close neighbour, got {gap_shuffled:.3f}"
assert gap_time < 0.05, \
    f"a time-ordered split should have almost no near-in-time neighbours, got {gap_time:.3f}"
assert gap_shuffled > 8 * gap_time, "the shuffled split should be dramatically leakier"
# The `within` parameter must actually be used.
_w0 = neighbour_gap_share(X_train.index, X_test.index, within=0)
_w50 = neighbour_gap_share(X_train.index, X_test.index, within=50)
assert _w0 < gap_shuffled < _w50, \
    "the `within` argument must widen or narrow the window - yours looks hard-coded"
print(f"✅ Correct!  {gap_shuffled:.1%} of shuffled test rows sit within 2 days of a training",
      f"row, against {gap_time:.1%} under a time-ordered split - that gap IS the leak")

## 5. Purging and embargo

The time-ordered split fixed the big problem. It still has a small one at the seam.

The last training row is day *split−1*, and its target covers the following 5 days, which are the **first rows of the test set**. That training label was computed from test-period prices. It is a small leak, but it is the same kind, and the fix costs nothing.

**Purging** removes training rows whose target window overlaps the test set. **Embargo** additionally drops a few rows after the boundary so that slow features built from training-period data do not bleed across.

The rule of thumb: purge at least `horizon` rows. When features use long lookbacks, embargo a few more.

```
train ................................  [purge]  test ...............
                                        ^^^^^^^
                                  these rows' targets reach
                                  into the test period
```

In [12]:
def purged_split(X, y, train_frac=0.7, horizon=HORIZON, embargo=0):
    cut = int(len(X) * train_frac)
    train_end = cut - horizon - embargo     # drop the contaminated tail

    return (X.iloc[:train_end], X.iloc[cut:],
            y.iloc[:train_end], y.iloc[cut:])


Xp_tr, Xp_te, yp_tr, yp_te = purged_split(X, y, embargo=5)

print(f"plain time split : train {len(X_tr)} rows, test {len(X_te)} rows")
print(f"purged + embargo : train {len(Xp_tr)} rows, test {len(Xp_te)} rows "
      f"({len(X_tr) - len(Xp_tr)} rows removed at the seam)")

m = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xp_tr, yp_tr)
print(f"test accuracy    : {m.score(Xp_te, yp_te):.3f}")

plain time split : train 1016 rows, test 436 rows
purged + embargo : train 1006 rows, test 436 rows (10 rows removed at the seam)


test accuracy    : 0.511


Ten rows out of a thousand, and the accuracy barely moves, which is the honest result to report. Purging is not what rescues this model. It is a correctness detail that matters much more when the horizon is long relative to the sample, and it costs so little that there is no reason to skip it.

The lesson to take is the ordering: fix the split first, because that is worth 21 points. Then fix the seam, because it is nearly free.

### ✏️ Your turn: purge the seam yourself

Write two functions.

`purged_time_split(X, y, train_frac=0.7, horizon=5, embargo=0)` returns the tuple `(X_tr, X_te, y_tr, y_te)`, where the test block starts at `cut = int(len(X) * train_frac)` and the training block **ends** `horizon + embargo` rows before it.

`overlap_count(n_train, test_start, horizon)` returns how many of the training rows `0 … n_train-1` have a target window reaching the test block. That is, how many satisfy `i + horizon >= test_start`.

Then set `overlap_naive` to the count for an **unpurged** split (training rows `0 … cut-1`) and `overlap_purged` to the count for your purged split. The first should be positive; the second must be `0`.

In [ ]:
def purged_time_split(X, y, train_frac=0.7, horizon=5, embargo=0):
    cut = int(len(X) * train_frac)
    train_end = ...          # TODO: stop short of the test block
    return ...


def overlap_count(n_train, test_start, horizon):
    # TODO: how many training rows have a target window reaching test_start?
    ...


cut = int(len(X) * 0.7)
Xq_tr, Xq_te, yq_tr, yq_te = purged_time_split(X, y, 0.7, HORIZON, embargo=0)

overlap_naive = ...
overlap_purged = ...

print(f"train {len(Xq_tr)} test {len(Xq_te)} | "
      f"contaminated rows: naive {overlap_naive}, purged {overlap_purged}")

In [ ]:
_cut = int(len(X) * 0.7)
assert len(Xq_tr) == _cut - HORIZON, \
    f"with embargo=0 the training block should end {HORIZON} rows early, got {len(Xq_tr)} vs {_cut - HORIZON}"
assert len(Xq_te) == len(X) - _cut, "the test block should start at the cut and run to the end"
assert Xq_te.index[0] == X.index[_cut], "the test block must begin exactly at the cut"
assert len(yq_tr) == len(Xq_tr) and len(yq_te) == len(Xq_te), "X and y blocks must line up"
assert overlap_purged == 0, \
    f"a purged split must leave 0 contaminated training rows, got {overlap_purged}"
assert overlap_naive == HORIZON, \
    f"an unpurged split leaves exactly {HORIZON} contaminated rows, got {overlap_naive}"
# The embargo must widen the gap further.
_e = purged_time_split(X, y, 0.7, HORIZON, embargo=7)[0]
assert len(_e) == _cut - HORIZON - 7, "the embargo argument must shrink the training block further"
# No training row may sit at or past the cut.
assert Xq_tr.index[-1] < Xq_te.index[0], "training data must end strictly before the test block"
print(f"✅ Correct!  Purging removed the {overlap_naive} rows whose target windows",
      f"reached into the test block, leaving {overlap_purged}.")

## Part 2: Getting a number you can believe

## 6. Overfitting, and why training accuracy is not a number

Both models above hit **1.000** on their training data. A model that is perfect on data it has seen and coin-flip on data it has not has not learned a pattern; it has learned the answer key.

A decision tree can keep splitting until every training row sits alone in its own leaf, at which point it has recorded the dataset rather than generalised from it. `min_samples_leaf` stops that: it forbids any leaf holding fewer than *k* rows, so the tree must find rules covering *k* rows at once, and rules that cover many rows are the ones that might also cover future rows.

Watch what constraining the model does to both numbers.

In [15]:
rows = []
for leaf in [1, 5, 20, 50, 100, 200]:
    m = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, random_state=0)
    m.fit(Xp_tr, yp_tr)
    rows.append({
        "min_samples_leaf": leaf,
        "train_acc": round(m.score(Xp_tr, yp_tr), 3),
        "test_acc": round(m.score(Xp_te, yp_te), 3),
        "gap": round(m.score(Xp_tr, yp_tr) - m.score(Xp_te, yp_te), 3),
    })

pd.DataFrame(rows)

,min_samples_leaf,train_acc,test_acc,gap
0,1,1.000,0.511,0.489
1,5,0.943,0.502,0.441
2,20,0.772,0.532,0.240
3,50,0.678,0.544,0.134
4,100,0.632,0.530,0.102
5,200,0.601,0.564,0.037


Read the columns separately, because they say different things.

**`train_acc` falls monotonically**, 1.000 → 0.601, and the **`gap` collapses** from 0.489 to 0.037. That is the constraint working exactly as intended: the model is progressively forbidden from memorising.

**`test_acc` rises, but raggedly**: 0.511, 0.502, 0.532, 0.544, 0.530, 0.564. It is higher at the tight end than the loose end, which is the real lesson: *weakening the model improved it out of sample.*

> 🧠 **The train–test gap is the diagnostic.** A large gap means the model memorised. Closing it by *weakening* the model is the normal, correct move, and it feels wrong the first several times you do it.

Now resist an obvious temptation. `min_samples_leaf=200` tops the table, so why not use it? Because that column is 436 test rows, and the wobble in it, 0.544 down to 0.530 and back up to 0.564, is larger than most of the differences you would be choosing between. Picking the winner off this table means selecting on noise, which is section 12.1's error committed in advance.

We carry **`min_samples_leaf=20`** forward as a moderate, chosen-in-advance default, not as the table's champion. Section 7 introduces the machinery that could settle the question properly.

### ✏️ Your turn: the overfitting diagnostic

Write `complexity_curve(X_tr, y_tr, X_te, y_te, leaves)` returning a DataFrame with one row per value in `leaves` and exactly these columns, in order:

| column | value |
|---|---|
| `min_samples_leaf` | the value from `leaves` |
| `train_acc` | accuracy on the training block |
| `test_acc` | accuracy on the test block |
| `gap` | `train_acc − test_acc` |

Fit `RandomForestClassifier(n_estimators=100, min_samples_leaf=leaf, random_state=0)` for each. Do **not** round. The check compares exact values. (100 trees, not the 200 used in the table above, so your numbers will differ slightly from it. The shape is what matters.)

Run it on the purged blocks with `leaves=[1, 10, 50, 200]` as `curve`, then set `gap_shrinks` to `True` if the `gap` column decreases at every step.

In [ ]:
def complexity_curve(X_tr, y_tr, X_te, y_te, leaves):
    rows = []
    for leaf in leaves:
        # TODO: fit, score both blocks, append one row
        ...

    return pd.DataFrame(rows)


curve = complexity_curve(Xp_tr, yp_tr, Xp_te, yp_te, [1, 10, 50, 200])
gap_shrinks = ...

print(curve.round(4).to_string(index=False))
print(f"gap shrinks at every step: {gap_shrinks}")

In [ ]:
COLS = ["min_samples_leaf", "train_acc", "test_acc", "gap"]
assert list(curve.columns) == COLS, f"columns must be exactly {COLS} in order"
assert list(curve["min_samples_leaf"]) == [1, 10, 50, 200], "one row per leaf value, in order"
assert np.allclose(curve["gap"], curve["train_acc"] - curve["test_acc"]), \
    "gap must be train_acc - test_acc"
# Recompute independently.
_exp = []
for _leaf in [1, 10, 50, 200]:
    _m = RandomForestClassifier(n_estimators=100, min_samples_leaf=_leaf, random_state=0)
    _m.fit(Xp_tr, yp_tr)
    _exp.append((float(_m.score(Xp_tr, yp_tr)), float(_m.score(Xp_te, yp_te))))
assert np.allclose(curve["train_acc"], [e[0] for e in _exp]), \
    "train_acc is off - fit on Xp_tr/yp_tr with n_estimators=100 and random_state=0"
assert np.allclose(curve["test_acc"], [e[1] for e in _exp]), "test_acc is off"
assert np.isclose(curve["train_acc"].iloc[0], 1.0), \
    "an unconstrained forest should memorise its training set perfectly"
assert gap_shrinks is True, "the gap should shrink at every step as the constraint tightens"
assert curve["test_acc"].iloc[-1] > curve["test_acc"].iloc[0], \
    "the most-constrained model should beat the unconstrained one out of sample"
print(f"✅ Correct!  The gap falls {curve['gap'].iloc[0]:.3f} -> {curve['gap'].iloc[-1]:.3f}",
      f"while test accuracy rises {curve['test_acc'].iloc[0]:.3f} -> {curve['test_acc'].iloc[-1]:.3f}.",
      "Weakening the model improved it.")

## 7. Walk-forward validation

One time-ordered split gives you one number from one period. Module 6 section 8.2 made the point with rolling Sharpe: a single figure hides everything about consistency, and a single test period might just be a good year.

**Walk-forward** validation is the fix, and it is what actual trading firms use. Train on everything up to a point, test on the block that follows, roll forward, repeat:

```
fold 1:  train[========]  test[==]
fold 2:  train[==========]  test[==]
fold 3:  train[============]  test[==]
```

The training set grows; the test block always sits in the future relative to its own training data. You end up with an accuracy per fold, which is a distribution rather than a point, and the spread of that distribution is usually more informative than its mean.

In [18]:
def walk_forward(X, y, n_folds=6, start_frac=0.4, leaf=20, horizon=HORIZON):
    start = int(len(X) * start_frac)
    step = (len(X) - start) // n_folds

    accs = []
    predictions = pd.Series(index=X.index, dtype=float)

    for f in range(n_folds):
        tr_end = start + f * step
        te_end = min(tr_end + step, len(X))

        m = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf,
                                   random_state=0)
        m.fit(X.iloc[:tr_end - horizon], y.iloc[:tr_end - horizon])   # purged

        p = m.predict(X.iloc[tr_end:te_end])
        predictions.iloc[tr_end:te_end] = p
        accs.append(float((p == y.iloc[tr_end:te_end].values).mean()))

    return np.array(accs), predictions


accs, predictions = walk_forward(X, y)

for i, a in enumerate(accs, 1):
    print(f"  fold {i}: {a:.3f}")
print(f"\nmean {accs.mean():.3f}   std {accs.std():.3f}   "
      f"range {accs.min():.3f} to {accs.max():.3f}")

  fold 1: 0.517
  fold 2: 0.655
  fold 3: 0.662
  fold 4: 0.559
  fold 5: 0.559
  fold 6: 0.497

mean 0.575   std 0.063   range 0.497 to 0.662


**Mean 0.575, and folds ranging from 0.497 to 0.662.**

Both numbers matter and the second one matters more than people expect. Had you run a single split and landed on fold 3, you would be reporting 66% and planning your fund. Had you landed on fold 6, you would have concluded the idea was dead. Neither would have been wrong about its own fold, and both would have been badly misleading.

That spread is not noise to be averaged away and forgotten. It is the honest statement of how much you know: *somewhere around 57%, and it varies a lot.*

### ✏️ Your turn: walk-forward validation

Write `my_walk_forward(X, y, n_folds, leaf, horizon)` returning a **list** of per-fold accuracies, following the scheme above:

1. `start = len(X) // 2`: the first fold trains on the first half.
2. `step = (len(X) - start) // n_folds`.
3. For fold `f`, train on rows `0` to `start + f*step − horizon` (the purge), and test on rows `start + f*step` to `start + (f+1)*step`, capped at `len(X)`.
4. Use `RandomForestClassifier(n_estimators=100, min_samples_leaf=leaf, random_state=0)`.

Then run it with `n_folds=5, leaf=20, horizon=5` and set `wf_scores` to the list, `wf_mean` to its mean and `wf_std` to its standard deviation (both floats).

In [ ]:
def my_walk_forward(X, y, n_folds=5, leaf=20, horizon=5):
    start = len(X) // 2
    step = (len(X) - start) // n_folds
    scores = []

    for f in range(n_folds):
        # TODO: fit on the purged history, score on the next block
        ...

    return scores


wf_scores = my_walk_forward(X, y, n_folds=5, leaf=20, horizon=5)
wf_mean = ...
wf_std = ...

print([round(s, 3) for s in wf_scores])
print(f"mean {wf_mean:.3f} std {wf_std:.3f}")

In [ ]:
assert len(wf_scores) == 5, f"expected 5 fold scores, got {len(wf_scores)}"
assert all(0.0 <= s <= 1.0 for s in wf_scores), "accuracies must be between 0 and 1"
assert np.isclose(wf_mean, float(np.mean(wf_scores))), "wf_mean should be the mean of the folds"
assert np.isclose(wf_std, float(np.std(wf_scores))), "wf_std should be the std of the folds"
# Recompute independently.
_s = len(X) // 2
_step = (len(X) - _s) // 5
_exp = []
for _f in range(5):
    _a = _s + _f * _step
    _b = min(_a + _step, len(X))
    _m = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=0)
    _m.fit(X.iloc[:_a - 5], y.iloc[:_a - 5])
    _exp.append(float((_m.predict(X.iloc[_a:_b]) == y.iloc[_a:_b].values).mean()))
assert np.allclose(wf_scores, _exp), (
    "fold scores do not match - check that you train on rows 0 to (tr_end - horizon) "
    "and test on tr_end to te_end")
assert wf_std > 0.01, "the folds should genuinely differ from each other"
print(f"✅ Correct!  folds {[round(s, 3) for s in wf_scores]},",
      f"mean {wf_mean:.3f}, std {wf_std:.3f} - the spread is the honest uncertainty")

## 8. The baseline: is 57.5% good?

Accuracy has no meaning without something to compare it to. If a market rises on 57% of periods, a model that always predicts "up" scores 57% while knowing nothing at all.

Always report the **majority-class baseline** next to your accuracy. It is the score of the dumbest possible model.

In [21]:
evaluated = predictions.dropna()
truth = y.loc[evaluated.index]

majority = max(truth.mean(), 1 - truth.mean())

print(f"model accuracy    : {(evaluated == truth).mean():.3f}")
print(f"majority baseline : {majority:.3f}  (always predict "
      f"{'up' if truth.mean() > 0.5 else 'down'})")
print(f"edge over baseline: {(evaluated == truth).mean() - majority:+.3f}")

model accuracy    : 0.575
majority baseline : 0.508  (always predict up)
edge over baseline: +0.067


About seven points of edge over the do-nothing baseline. That is a real result on this data, and it is worth being precise about what "real" means here, because we built the data ourselves and put a persistent regime in it. The model found something we know is there. On real market data the honest expectation is a fraction of this, and frequently nothing.

### Where the errors fall

Accuracy compresses four different outcomes into one number. The **confusion matrix** shows all four, and they are not interchangeable: predicting "up" when the market falls costs you money, while predicting "down" when it rises merely costs you an opportunity.

In [22]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(truth, evaluated)
frame = pd.DataFrame(cm,
                     index=["actually down", "actually up"],
                     columns=["predicted down", "predicted up"])
print(frame)
print()

tn, fp, fn, tp = cm.ravel()
print(f"when it said UP   it was right {tp / (tp + fp):.1%} of the time "
      f"({tp + fp} calls)")
print(f"when it said DOWN it was right {tn / (tn + fn):.1%} of the time "
      f"({tn + fn} calls)")

               predicted down  predicted up
actually down             272           156
actually up               214           228

when it said UP   it was right 59.4% of the time (384 calls)
when it said DOWN it was right 56.0% of the time (486 calls)


The two sides are close here, roughly 59% on up-calls against 56% on down-calls, but that is a fact you had to *check*, not one accuracy could have told you. Had the split been 65/48, the sensible strategy would be to trade the strong side and stay flat on the weak one rather than going short on a signal that is barely better than a coin.

Notice also the raw counts: 384 up-calls against 486 down-calls, on a sample that actually rose 50.8% of the time. The model leans short more often than the market falls, and still comes out ahead. Accuracy alone hides that entirely.

> 🧠 **In sklearn's confusion matrix, rows are truth and columns are predictions.** Getting them backwards is a rite of passage; check with a case you can verify by hand.

### ✏️ Your turn: model versus baseline

Write `baseline_report(y_true, y_pred)` returning a dict with exactly these keys:

| key | value |
|---|---|
| `accuracy` | share of correct predictions |
| `baseline` | the majority-class share of `y_true` |
| `edge` | `accuracy − baseline` |
| `up_precision` | of the rows predicted `1`, the share that were truly `1` |
| `n` | number of rows, an int |

If the model never predicts `1`, `up_precision` must be `float('nan')` rather than raising a `ZeroDivisionError`.

Apply it to the walk-forward predictions as `report`, using `truth` and `evaluated`.

In [ ]:
def baseline_report(y_true, y_pred):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)

    accuracy = ...
    baseline = ...
    n_up_calls = ...

    return {
        "accuracy": ...,
        "baseline": ...,
        "edge": ...,
        "up_precision": ...,
        "n": ...,
    }


report = baseline_report(truth, evaluated)
report

In [ ]:
KEYS = ["accuracy", "baseline", "edge", "up_precision", "n"]
assert list(report) == KEYS, f"keys must be exactly {KEYS} in order"
assert np.isclose(report["accuracy"], float((truth.values == evaluated.values).mean())), \
    "accuracy is off"
assert np.isclose(report["baseline"], float(max(truth.mean(), 1 - truth.mean()))), \
    "baseline should be the MAJORITY class share, i.e. max(p, 1-p)"
assert np.isclose(report["edge"], report["accuracy"] - report["baseline"]), "edge is off"
assert report["n"] == len(truth), "n should be the number of rows"
# Hand-checkable case.
_t = pd.Series([1, 1, 1, 0])
_p = pd.Series([1, 1, 0, 0])
_r = baseline_report(_t, _p)
assert np.isclose(_r["accuracy"], 0.75), f"expected accuracy 0.75 on the hand case, got {_r['accuracy']}"
assert np.isclose(_r["baseline"], 0.75), f"expected baseline 0.75, got {_r['baseline']}"
assert np.isclose(_r["up_precision"], 1.0), f"expected up_precision 1.0, got {_r['up_precision']}"
# Never-predicts-up must not raise.
_r2 = baseline_report(pd.Series([1, 0, 1]), pd.Series([0, 0, 0]))
assert np.isnan(_r2["up_precision"]), \
    "when the model never predicts 1, up_precision must be NaN, not an error or 0"
print(f"✅ Correct!  accuracy {report['accuracy']:.3f} against a {report['baseline']:.3f}",
      f"baseline - an edge of {report['edge']:+.3f}")

## Part 3: From a model to a strategy

## 9. Accuracy is not the goal

A model can be accurate and unprofitable, or inaccurate and profitable. Accuracy weights every day equally; your P&L weights days by how much the market moved. Being right on fifty quiet days and wrong on one violent one is a good accuracy score and a bad month.

Module 6 built the tools for the question that actually matters. Convert predictions into positions and run them.

The conversion: predict `1` → hold long; predict `0` → hold short. And the same causality rule as always, `shift(1)`, because a prediction made using today's close can only be traded from tomorrow.

In [25]:
position = (evaluated * 2 - 1).rename("position")        # 1 -> +1, 0 -> -1
market = price.pct_change().reindex(evaluated.index)

strategy_returns = (position.shift(1) * market).dropna()


def performance(r, label):
    eq = (1 + r).cumprod()
    ann_vol = r.std() * np.sqrt(252)
    return {
        "strategy": label,
        "total": f"{eq.iloc[-1] - 1:.2%}",
        "sharpe": round(float((r.mean() * 252) / ann_vol), 3),
        "max_dd": f"{(eq / eq.cummax() - 1).min():.2%}",
    }


buy_hold = market.dropna()
pd.DataFrame([performance(strategy_returns, "ML walk-forward"),
              performance(buy_hold, "buy and hold")])

,strategy,total,sharpe,max_dd
0,ML walk-forward,201.35%,1.865,-14.63%
1,buy and hold,16.79%,0.338,-44.56%


A Sharpe of 1.865 against 0.338 for buying and holding, on out-of-sample predictions only.

**Now be suspicious of that number**, because it is exactly the kind of result Module 6 taught you to distrust, and here we have the unusual luxury of knowing why it is as high as it is. We built this market with a persistent, learnable regime. The model found it. A real market does not contain a clean two-state drift that a random forest can recover from six momentum features, and the same pipeline on real data would produce something far more modest.

The pipeline is the lesson. The Sharpe is a property of the simulation.

There is also one thing conspicuously missing from that table.

## 10. Costs, again

A model that flips between long and short is trading. Module 6 section 6 showed that a strategy is only real if it survives its own transaction costs, and a 5-day-horizon classifier changes its mind often.

In [26]:
turnover = position.diff().abs().fillna(0.0)          # 2.0 on a full flip

print(f"position changes : {int((position.diff() != 0).sum())} "
      f"over {len(position)} days")
print(f"average turnover : {turnover.mean():.3f} per day")
print()

rows = []
for bps in [0, 5, 10, 20, 35, 50]:
    net = strategy_returns - turnover.reindex(strategy_returns.index).fillna(0) * bps / 10_000
    eq = (1 + net).cumprod()
    rows.append({
        "cost_bps": bps,
        "total": f"{eq.iloc[-1] - 1:.2%}",
        "sharpe": round(float((net.mean() * 252) / (net.std() * np.sqrt(252))), 3),
    })

pd.DataFrame(rows)

position changes : 119 over 870 days
average turnover : 0.271 per day



,cost_bps,total,sharpe
0,0,201.35%,1.865
1,5,167.81%,1.673
2,10,137.97%,1.481
3,20,87.84%,1.095
4,35,31.61%,0.523
5,50,-7.89%,-0.031


The edge is gone somewhere around 50 basis points per unit of turnover.

Whether that is comfortable depends entirely on what you trade. On a liquid US equity or a major ETF, a few basis points is realistic and this strategy survives easily. On a small-cap name with a wide spread, 50bp is optimistic and the strategy is fictional. **The same model is a business or a waste of time depending on the instrument**, and nothing in the accuracy score tells you which.

> 🧠 **Report the breakeven cost, not just the gross Sharpe.** It converts an abstract result into a question with an answer: *can I actually trade this cheaply enough?*

### ✏️ Your turn: predictions to a strategy

Write `strategy_from_predictions(pred, px, cost_bps=0)` that:

1. maps predictions to positions (`1 → +1`, `0 → −1`),
2. computes daily returns as `position.shift(1) × px.pct_change()` over `pred`'s dates,
3. subtracts `turnover × cost_bps / 10000`, where turnover is `position.diff().abs()` filled with `0`,
4. returns the net daily return Series with `NaN`s dropped.

Then write `breakeven_bps(pred, px)` returning the **smallest whole** number of basis points, searching `0, 1, 2, …, 200`, at which the annualized Sharpe of the net returns drops to `0` or below. Return `200` if it never does.

Set `net_10` to the strategy at 10bp and `be` to the breakeven.

In [ ]:
def strategy_from_predictions(pred, px, cost_bps=0):
    position = pred * 2 - 1
    # TODO: gross returns, then subtract turnover cost
    ...


def breakeven_bps(pred, px):
    for bps in range(0, 201):
        r = strategy_from_predictions(pred, px, bps)
        sharpe = ...
        if ...:
            return bps
    return 200


net_10 = strategy_from_predictions(evaluated, price, 10)
be = breakeven_bps(evaluated, price)

print(f"10bp Sharpe {(net_10.mean() * 252) / (net_10.std() * np.sqrt(252)):.3f} "
      f"| breakeven {be} bps")

In [ ]:
_pos = evaluated * 2 - 1
_mkt = price.pct_change().reindex(evaluated.index)
_gross = (_pos.shift(1) * _mkt)
_turn = _pos.diff().abs().fillna(0.0)
_exp10 = (_gross - _turn * 10 / 10_000).dropna()
assert np.allclose(net_10.values, _exp10.values), \
    "net_10 is off - shift the position by one day and charge turnover * bps/10000"
_exp0 = strategy_from_predictions(evaluated, price, 0)
assert np.allclose(_exp0.values, _gross.dropna().values), \
    "at 0 bps the result must equal the gross returns"
assert isinstance(be, (int, np.integer)), "breakeven_bps must return a whole number"
assert 20 < be < 200, f"the breakeven should be a realistic double-digit figure, got {be}"
_at = strategy_from_predictions(evaluated, price, be)
_before = strategy_from_predictions(evaluated, price, be - 1)
assert (_at.mean() * 252) / (_at.std() * np.sqrt(252)) <= 0, \
    "at the breakeven the Sharpe must be <= 0"
assert (_before.mean() * 252) / (_before.std() * np.sqrt(252)) > 0, \
    "one bp below the breakeven the Sharpe must still be positive - return the SMALLEST such bps"
print(f"✅ Correct!  The edge survives to {int(be)} bps of cost per unit of turnover.",
      "Below that it is a strategy; above it, it is a spreadsheet.")

## 11. Feature importance, and what it does not mean

`feature_importances_` reports how much each feature reduced impurity across the forest's splits. It is useful and it is routinely over-read.

In [29]:
final = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=0)
final.fit(Xp_tr, yp_tr)

imp = pd.Series(final.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.round(4))
print()

# Does dropping the top feature actually hurt? That is the real test.
top = imp.index[0]
reduced = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=0)
reduced.fit(Xp_tr.drop(columns=top), yp_tr)

print(f"all features            : {final.score(Xp_te, yp_te):.3f}")
print(f"without '{top}' : {reduced.score(Xp_te.drop(columns=top), yp_te):.3f}")

mom_20       0.2577
ma_ratio     0.1652
mom_60       0.1645
mom_5        0.1511
vol_20       0.1485
range_pos    0.1130
dtype: float64



all features            : 0.532
without 'mom_20' : 0.518


Two things in that output are worth more than the ranking itself.

The importances are **flat**, 0.258 at the top down to 0.113 at the bottom, across six features. There is no dominant signal, only a committee of weak ones, which is the normal state of affairs in financial ML and the reason ensembles are used at all.

And dropping the top-ranked feature cost **1.4 points** of test accuracy. Not nothing, but far less than "most important feature" implies, because `mom_5`, `mom_20` and `mom_60` are three views of the same quantity. Remove one and the others cover for it.

Three cautions, in descending order of how often they bite:

1. **Importance is not causation, and not even predictive power.** It measures how often the forest *used* a feature, which is influenced by how many distinct values the feature has. Continuous features look more important than binary ones almost automatically.
2. **Correlated features split the credit.** `mom_20` and `mom_60` carry overlapping information, so each gets a fraction of the importance that the pair jointly deserves, and, as the drop test just showed, each is also more expendable than its score suggests.
3. **It is computed on training data.** A feature the model leaned on to memorise noise still scores well.

The check that means something is the one in the cell above: **drop the feature, refit, and see whether out-of-sample performance falls.** That is expensive and it is the only version that answers the question you are actually asking.

## Part 4: Four ways this still goes wrong

### 12.1 You will try more than one model

The multiple-testing problem from Module 6 section 8.4 and Module 7 section 11.3 arrives here in its most tempting form. Trying twelve configurations and reporting the best is a search, and the best of twelve is biased upward whether or not any of them work.

In [30]:
configs = [(leaf, depth) for leaf in [5, 20, 50] for depth in [3, 5, 10, None]]

results = []
for leaf, depth in configs:
    m = RandomForestClassifier(n_estimators=100, min_samples_leaf=leaf,
                               max_depth=depth, random_state=0)
    m.fit(Xp_tr, yp_tr)
    results.append({"min_samples_leaf": leaf, "max_depth": depth,
                    "test_acc": round(m.score(Xp_te, yp_te), 4)})

table = pd.DataFrame(results).sort_values("test_acc", ascending=False)
print(f"configurations tried : {len(configs)}")
print(f"best  test accuracy  : {table['test_acc'].max():.4f}")
print(f"worst test accuracy  : {table['test_acc'].min():.4f}")
print(f"spread               : {table['test_acc'].max() - table['test_acc'].min():.4f}")
print()
print(table.head(4).to_string(index=False))

configurations tried : 12
best  test accuracy  : 0.5436
worst test accuracy  : 0.5069
spread               : 0.0367

 min_samples_leaf  max_depth  test_acc
               20       10.0    0.5436
               50       10.0    0.5390
               50        NaN    0.5390
               20        NaN    0.5367


Twelve configurations, and **3.7 percentage points between the best and the worst** on identical data. Some of that difference is real and some is luck, and this table cannot separate them, because every configuration was scored on the same test set, which makes that test set part of the training process.

Report only the winner, and you have reported a number selected for being the largest of twelve draws. Module 6 section 8.4 gave the general form of this: the maximum of many noisy estimates is biased upward even when every underlying estimate is worthless.

The discipline: choose hyperparameters on a **validation** period, then evaluate the chosen model once on a test period you have never looked at. Once you have looked, it is no longer a test set. And report how many configurations you tried, in the same way Module 6 insisted you report how many strategies you searched.

### 12.2 The relationship changes

Modules 6 and 7 both ended here, and ML is the most exposed of the three. A cointegrated pair can break; a trend rule can stop working; a model that has learned a mapping from features to returns is learning a relationship that markets actively compete away. The strongest patterns are the first to be arbitraged out.

Practical consequences: retrain on a schedule rather than fitting once; monitor live accuracy against the walk-forward distribution and stop when it falls outside it; prefer simpler models, which degrade more gracefully.

### 12.3 The features can still see the future

Look-ahead in ML is harder to spot than in a hand-written signal, because the model does not tell you *how* it got the answer. The usual sources:

- a `shift(-k)` in a feature, which is the direct version
- fundamentals stamped with the period they describe rather than the date they were published
- an index constituent list as of today, applied to five years ago (survivorship bias, Module 6 section 1)
- scaling with `StandardScaler().fit(X)` on the **whole** dataset before splitting. The scaler learns the full-sample mean and standard deviation, which is Module 7's full-sample z-score trap wearing a different hat

That last one is worth stating plainly because it is subtle and extremely common: fit the scaler on the training fold only, then apply it to the test fold.

### 12.4 More data is not more information

Six years of daily data is 1,512 rows, and section 4 showed that consecutive rows are near-copies. Switching to hourly bars gives you 24× the rows and nothing like 24× the information. Model complexity should be set by how much *independent* information you have, which is far less than `len(X)` suggests.

### ✏️ Your turn: the honest evaluation, end to end

Write `evaluate_model(X, y, px, leaf=20, n_folds=6, horizon=5, cost_bps=10)` that runs the whole pipeline and returns a dict with exactly these keys:

| key | value |
|---|---|
| `mean_accuracy` | mean of the walk-forward fold accuracies |
| `accuracy_std` | standard deviation of those accuracies |
| `baseline` | majority-class share over the evaluated rows |
| `edge` | `mean_accuracy − baseline` |
| `sharpe` | annualized Sharpe of the net strategy returns at `cost_bps` |
| `verdict` | `"tradeable"` if `edge > 0.02` **and** `sharpe > 0.5`, else `"no edge"` |

Use the `walk_forward` helper from section 7 for the folds and predictions, and your `strategy_from_predictions` for the returns.

Run it on `X, y, price` as `verdict_real`. Then run it on a **shuffled target**, `y_fake = pd.Series(np.random.default_rng(0).permutation(y.values), index=y.index)`, as `verdict_fake`. Destroying the link between features and target must destroy the verdict. If it does not, your pipeline is leaking.

In [ ]:
def evaluate_model(X, y, px, leaf=20, n_folds=6, horizon=5, cost_bps=10):
    accs, preds = walk_forward(X, y, n_folds=n_folds, leaf=leaf, horizon=horizon)
    ev = preds.dropna()
    tru = y.loc[ev.index]

    # TODO: assemble the six values
    ...

    return {
        "mean_accuracy": ..., "accuracy_std": ..., "baseline": ...,
        "edge": ..., "sharpe": ..., "verdict": ...,
    }


verdict_real = evaluate_model(X, y, price)

y_fake = pd.Series(np.random.default_rng(0).permutation(y.values), index=y.index)
verdict_fake = evaluate_model(X, y_fake, price)

print("real:", verdict_real)
print("fake:", verdict_fake)

In [ ]:
KEYS = ["mean_accuracy", "accuracy_std", "baseline", "edge", "sharpe", "verdict"]
assert list(verdict_real) == KEYS, f"keys must be exactly {KEYS} in order"
_a, _p = walk_forward(X, y, n_folds=6, leaf=20, horizon=5)
assert np.isclose(verdict_real["mean_accuracy"], float(np.mean(_a))), "mean_accuracy is off"
assert np.isclose(verdict_real["accuracy_std"], float(np.std(_a))), "accuracy_std is off"
assert np.isclose(verdict_real["edge"],
                  verdict_real["mean_accuracy"] - verdict_real["baseline"]), "edge is off"
assert verdict_real["verdict"] == "tradeable", \
    "on the real target this pipeline should find a tradeable edge"
# The scrambled target is the real test: no relationship can survive it.
assert verdict_fake["verdict"] == "no edge", (
    f"a randomly permuted target MUST come back 'no edge', got {verdict_fake}. "
    "If a shuffled target still looks tradeable, the pipeline is leaking.")
assert verdict_fake["edge"] < verdict_real["edge"], \
    "the scrambled target should show a smaller edge than the real one"
print(f"✅ Correct!  real -> {verdict_real['verdict']} "
      f"(edge {verdict_real['edge']:+.3f}, Sharpe {verdict_real['sharpe']:.2f});  "
      f"scrambled -> {verdict_fake['verdict']} (edge {verdict_fake['edge']:+.3f}).",
      "The shuffled-target check is the best single test of a pipeline.")

That last exercise contains the most valuable habit in this module, so it is worth naming.

> 🧠 **The scrambled-target test.** Destroy the relationship between your features and your target by permuting the target, then run your *entire* pipeline unchanged. It must report nothing. If it still finds an edge, the edge was coming from your pipeline rather than your data, and you have just caught a leak that no amount of staring at the code would have found.

Run it before you believe any result, including your own.

## 13. On QuantConnect

Research first: build the same feature matrix from real data in a QuantBook notebook.

In [ ]:
# 🔵 QC cell, feature engineering in the research environment
qb = QuantBook()
symbol = qb.add_equity("SPY", Resolution.DAILY).symbol

history = qb.history(symbol, 2000, Resolution.DAILY)
close = history.loc[symbol]["close"]

import numpy as np
import pandas as pd

daily = close.pct_change()
features = pd.DataFrame({
    "mom_5": close.pct_change(5),
    "mom_20": close.pct_change(20),
    "mom_60": close.pct_change(60),
    "vol_20": daily.rolling(20).std(),
    "range_pos": ((close - close.rolling(14).min())
                  / (close.rolling(14).max() - close.rolling(14).min())),
    "ma_ratio": close.rolling(10).mean() / close.rolling(50).mean(),
})

target = (close.shift(-5) / close - 1 > 0).astype(int)
data = features.join(target.rename("target")).dropna()

print(data.shape)
print(f"up-rate {data['target'].mean():.3f}")

Then the algorithm. The structure is Module 3's five pillars; the ML lives in a scheduled retrain, and the model is fitted **only** on data before the current moment.

In [ ]:
# 🔵 QC cell, an ML algorithm with scheduled retraining
class MLDirectionAlgorithm(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2016, 1, 1)
        self.set_end_date(2024, 1, 1)
        self.set_cash(100_000)
        self.set_brokerage_model(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE,
                                 AccountType.MARGIN)

        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol
        self.model = None
        self.lookback = 750          # ~3 years of training data
        self.horizon = 5

        self.set_warm_up(self.lookback + 60, Resolution.DAILY)

        # Retrain monthly. Never fit once in initialize and reuse forever.
        self.train(self.date_rules.month_start(self.symbol),
                   self.time_rules.at(8, 0),
                   self.fit_model)

    def build_features(self, close):
        daily = close.pct_change()
        return pd.DataFrame({
            "mom_5": close.pct_change(5),
            "mom_20": close.pct_change(20),
            "mom_60": close.pct_change(60),
            "vol_20": daily.rolling(20).std(),
            "range_pos": ((close - close.rolling(14).min())
                          / (close.rolling(14).max() - close.rolling(14).min())),
            "ma_ratio": close.rolling(10).mean() / close.rolling(50).mean(),
        })

    def fit_model(self):
        # self.history returns bars up to NOW - never the whole backtest.
        hist = self.history(self.symbol, self.lookback, Resolution.DAILY)
        if hist.empty:
            return

        close = hist.loc[self.symbol]["close"]
        feats = self.build_features(close)
        target = (close.shift(-self.horizon) / close - 1 > 0).astype(int)

        # Purge: the last `horizon` rows have targets reaching past today.
        frame = feats.join(target.rename("y")).dropna().iloc[:-self.horizon]
        if len(frame) < 200:
            return

        from sklearn.ensemble import RandomForestClassifier
        self.model = RandomForestClassifier(n_estimators=100, min_samples_leaf=20,
                                            random_state=0)
        self.model.fit(frame.drop(columns="y"), frame["y"])
        self.debug(f"{self.time.date()} retrained on {len(frame)} rows")

    def on_data(self, data: Slice):
        if self.is_warming_up or self.model is None:
            return
        if not data.bars.contains_key(self.symbol):
            return

        hist = self.history(self.symbol, 80, Resolution.DAILY)
        if hist.empty:
            return

        feats = self.build_features(hist.loc[self.symbol]["close"]).dropna()
        if feats.empty:
            return

        prediction = self.model.predict(feats.iloc[[-1]])[0]
        self.set_holdings(self.symbol, 1.0 if prediction == 1 else -1.0)

Three details in there are this module's lessons in executable form.

`self.train(...)` schedules the retrain on QuantConnect's training node, so a slow `fit` does not stall the algorithm's clock. Fitting once in `initialize` on all available history would be the look-ahead error from section 3, wearing its most respectable disguise.

`.iloc[:-self.horizon]` is section 5's purge: the final rows' targets depend on prices that have not happened yet at fit time.

`self.history(self.symbol, self.lookback, ...)` returns only bars up to now, which is what makes the whole thing causal.

> ⚠️ **A backtest of an ML algorithm is not out-of-sample if you chose the features by looking at the same period.** Every decision you made while exploring (which features, which horizon, which model) used the full history. The only genuinely clean test is forward in time, on data that did not exist when you designed it. Paper trading exists for this reason.

## Cheat sheet

**The pipeline**

| Task | Code |
|---|---|
| Features | backward-looking rolling stats only; **never** `shift(-k)` |
| Target | `(px.shift(-h) / px - 1 > 0).astype(int)`, `shift(-h)` allowed **here only** |
| Align | `features.join(target).dropna()` |
| Split | `X.iloc[:cut]`, `X.iloc[cut:]`, **never** `shuffle=True` |
| Purge | drop the last `horizon` training rows before the test block |
| Fit / predict | `m.fit(X_tr, y_tr)` · `m.predict(X_te)` · `m.score(X_te, y_te)` |
| Scale | fit `StandardScaler` on **train only**, then transform test |

**Models and diagnostics**

| Task | Code |
|---|---|
| Random forest | `RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=0)` |
| Curb overfitting | raise `min_samples_leaf`, lower `max_depth` |
| Overfit check | `train_acc − test_acc`; a large gap means memorisation |
| Majority baseline | `max(y.mean(), 1 - y.mean())` |
| Confusion matrix | `confusion_matrix(y_true, y_pred)`, rows truth, columns predictions |
| Importance | `m.feature_importances_`, then verify by dropping and refitting |

**Evaluation**

| Task | Code |
|---|---|
| Walk-forward | expanding train, next block as test, repeat; report mean **and** std |
| Positions | `position = pred * 2 - 1`, then `position.shift(1) * px.pct_change()` |
| Turnover cost | `position.diff().abs() * bps / 10_000` |
| Breakeven | smallest `bps` at which Sharpe ≤ 0 |
| **Scrambled-target test** | permute `y`, rerun everything; it must find nothing |

## Stretch goals

1. **Break your own pipeline.** Add a feature that peeks: `price.shift(-1) / price - 1`. Rerun `evaluate_model`. The accuracy should be near-perfect. Now find it using only the walk-forward output. This is what a leak looks like from the outside, and recognising the signature is the point.
2. **Logistic regression.** Fit `LogisticRegression(max_iter=2000)` on scaled features as a linear baseline. It is far less flexible than a forest; on data this noisy, does it do worse? Compare walk-forward means, not single splits.
3. **Regression instead of classification.** Predict the forward *return* with `RandomForestRegressor`, and size positions by the prediction rather than going fully long or short. Does variable sizing improve the Sharpe, or just the turnover?
4. **Horizon sweep.** Run the pipeline at horizons 1, 5, 10 and 20 days. Longer horizons have a better signal-to-noise ratio and fewer independent observations. Where is the sweet spot, and how confident can you be given the fold spread?
5. **The honest multiple-testing report.** You have now tried several models. Write down every configuration you ran across all of these exercises, then re-read Module 6 section 8.4 and state what your best result should be discounted by.
6. **On QuantConnect**, run `MLDirectionAlgorithm` on SPY, then on a single volatile stock. Compare the results against buy-and-hold in each case, and against the walk-forward accuracy distribution you would have predicted.

## What's next

**Module 9: Algorithm Framework, Risk Management and Competition Readiness** is where the eight modules become one submission. You will move from a monolithic `on_data` to QuantConnect's Algorithm Framework (alpha models, portfolio construction, execution and risk management as separate, testable pieces) add position sizing and drawdown controls, and assemble the checklist that separates a notebook experiment from something you would put your name on in a competition.

**Official docs:**
- [QuantConnect: applying research](https://www.quantconnect.com/docs/v2/research-environment/applying-research)
- [QuantConnect: training scheduled events](https://www.quantconnect.com/docs/v2/writing-algorithms/scheduled-events)
- [scikit-learn: cross-validation for time series](https://scikit-learn.org/stable/modules/cross_validation.html#time-series-split)
- [scikit-learn: RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

*MAT Education · ML & Capstone · Module 8.*